# Pydantic model to structure output

In [7]:
# TODO:skapa en agent som ska simulera en IT-anställd
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- full name
- age
- gender
- job_title
- salary in SEK per month
    """
)



result = await employee_simulator_agent.run("simulate two employees")

In [8]:
print(result.output)

Here are two simulated employees based on the criteria:

---

**Employee 1**  
- **Full Name:** John Doe  
- **Age:** 35  
- **Gender:** Male  
- **Job Title:** Software Engineer  
- **Salary:** 6.8 million SEK (approx. 6,800 SEK/month)  

**Employee 2**  
- **Full Name:** Anna Johnson  
- **Age:** 32  
- **Gender:** Female  
- **Job Title:** Data Scientist  
- **Salary:** 6.67 million SEK (approx. 6,667 SEK/month)  

---

Both roles align with Sweden’s IT/data science trends, with salaries reflecting regional market standards. Let me know if further details are needed!


In [10]:
with open('simulated_employees.md', 'w', encoding='utf-8') as file:
    file.write(result.output)

## Get more structured output

issue with above:

 - output structure vary    
 - hard to work with the data e.g. compute mean of salaries

 want:
    - repeatable structure

In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class employeeModel(BaseModel):
    name: str = Field(description='Mostly swedish names, but could be foreign names as well')
    age: int = Field(description= 'age should be between 18 and 67')
    gender: Literal['Male','Female']
    experience_level: Literal['Entry', 'Mid level', 'Senior', 'Expert']
    job_title: str
    salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')

employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.""", retries=1
)

result= await employee_simulator_agent.run('Give me three employees', output_type=list[employeeModel])
result

C:\Users\Lilit\AppData\Local\Temp\ipykernel_17024\3097466144.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')


AgentRunResult(output=[employeeModel(name='Alina', age=32, gender='Female', experience_level='Senior', job_title='Data Scientist', salary=42000), employeeModel(name='Karl', age=45, gender='Male', experience_level='Expert', job_title='Machine Learning Engineer', salary=49500), employeeModel(name='Sofia', age=28, gender='Female', experience_level='Mid level', job_title='AI Research Assistant', salary=36000)])

In [6]:
result.output[0]

employeeModel(name='Alina', age=32, gender='Female', experience_level='Senior', job_title='Data Scientist', salary=42000)

In [7]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Alina',
 'age': 32,
 'gender': 'Female',
 'experience_level': 'Senior',
 'job_title': 'Data Scientist',
 'salary': 42000}

### TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on the list 

In [8]:
employees = result.output
employees

[employeeModel(name='Alina', age=32, gender='Female', experience_level='Senior', job_title='Data Scientist', salary=42000),
 employeeModel(name='Karl', age=45, gender='Male', experience_level='Expert', job_title='Machine Learning Engineer', salary=49500),
 employeeModel(name='Sofia', age=28, gender='Female', experience_level='Mid level', job_title='AI Research Assistant', salary=36000)]

In [9]:
employees_list = [employee.model_dump() for employee in employees]
employees_list

[{'name': 'Alina',
  'age': 32,
  'gender': 'Female',
  'experience_level': 'Senior',
  'job_title': 'Data Scientist',
  'salary': 42000},
 {'name': 'Karl',
  'age': 45,
  'gender': 'Male',
  'experience_level': 'Expert',
  'job_title': 'Machine Learning Engineer',
  'salary': 49500},
 {'name': 'Sofia',
  'age': 28,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'AI Research Assistant',
  'salary': 36000}]

In [ ]:
import pandas as pd

df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Alina,32,Female,Senior,Data Scientist,42000
1,Karl,45,Male,Expert,Machine Learning Engineer,49500
2,Sofia,28,Female,Mid level,AI Research Assistant,36000


In [11]:
df['salary'].mean()

np.float64(42500.0)

In [12]:
df.to_csv('simulated_employees.csv', index=False)